In [1]:
# Standard library
import warnings
from operator import itemgetter

# Third-party
import bs4
from dotenv import load_dotenv
from pydantic import BaseModel, Field, PydanticDeprecatedSince20

# LangChain - core
from langchain_core.load import dumps, loads
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.documents import Document

# LangChain - integrations
from langchain_chroma.vectorstores import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangSmith
from langsmith import Client

# Suppress Pydantic v2 deprecation warnings from LangChain internals
warnings.filterwarnings("ignore", category=PydanticDeprecatedSince20)

load_dotenv()

/tmp/ipykernel_7222/3989172771.py:19: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


True

In [5]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = Groq(model="llama-3.1-8b-instant")
Settings.embed_model = HuggingFaceEmbedding()

documents = SimpleDirectoryReader("data_for_llama_indexing").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()
response = query_engine.query("What is the first article?")
print(response)

/home/roman/Desktop/RAG/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


The first article is titled "Retrieval-augmented generation."


In [5]:
import os.path
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

# check if storage already exists
PERSIST_DIR = "./storage"

if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("data_for_llama_indexing").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

# Either way we can now query the index
query_engine = index.as_query_engine()
response = query_engine.query("What is the third article about?")
print(response)

The third article discusses the limitations of LLMs and RAG. It highlights the potential for LLMs to provide incorrect information, even when using RAG to retrieve factually correct sources. The article also explains how LLMs can misinterpret context and generate misinformation, and how RAG can be used to prevent some of these errors by prioritizing new information.


In [6]:
# LlamaParse is async-first, so we need to run this line of code if you are working on a notebook
import nest_asyncio

nest_asyncio.apply()

In [6]:
from llama_cloud_services import LlamaParse

documents = LlamaParse(result_type="text", api_key=os.getenv("LLAMA_CLOUD_API_KEY")).load_data("./data_for_llama_indexing/RAG.txt")
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()
response = query_engine.query("What is the first article?")
print(response)

Started parsing the file under job_id 8bf6e8b1-9a4b-49c4-a4f6-7e4bfa78a4b2
The first article mentioned in the context information is "Barack Hussein Obama: America's First Muslim President?"
